<a href="https://colab.research.google.com/github/ananyavj/DeepLearning/blob/main/Question%204/D_Tweet_Recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/DeepLearning/datasets"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install gensim

In [ ]:
import pandas as pd

df = pd.read_csv(f"{BASE}/CBOW/twitter_dataset.csv")

print("Shape : ", df.shape)
print("Columns : ", df.columns.tolist())
print("First few rows under Text: ", df["Text"].head(5))
print("Missing Tweets : ", df["Text"].isnull().sum())

Shape :  (10000, 6)
Columns :  ['Tweet_ID', 'Username', 'Text', 'Retweets', 'Likes', 'Timestamp']
First few rows under Text:  0    Party least receive say or single. Prevent pre...
1    Hotel still Congress may member staff. Media d...
2    Nice be her debate industry that year. Film wh...
3    Laugh explain situation career occur serious. ...
4    Involve sense former often approach government...
Name: Text, dtype: object
Missing Tweets :  0


In [ ]:
import re
import numpy as np
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
def clean(text):
  text = text.lower()
  text = re.sub(r"<.*?>", " ", text)
  words = re.findall(r"[a-z]+", text)
  return [w for w in words if w not in ENGLISH_STOP_WORDS]

docs = [clean(tweet) for tweet in df["Text"]]

print("Length of doc : ", len(docs))
print("Docs preview : ", docs[:3])

Length of doc :  10000
Docs preview :  [['party', 'receive', 'say', 'single', 'prevent', 'prevent', 'husband', 'affect', 'cup', 'style', 'evening', 'protect', 'effect', 'stage', 'perform', 'possible', 'try', 'tax', 'share', 'style', 'television', 'successful', 'sell', 'development', 'economy', 'effect'], ['hotel', 'congress', 'member', 'staff', 'media', 'draw', 'buy', 'fly', 'identify', 'turn', 'minute', 'local', 'subject', 'way', 'believe', 'question', 'message', 'imagine', 'join', 'agency', 'indicate'], ['nice', 'debate', 'industry', 'year', 'film', 'generation', 'push', 'discover', 'partner', 'level', 'nearly', 'money', 'store', 'style', 'enjoy', 'kid', 'discuss', 'blue', 'save', 'model', 'everybody', 'especially', 'dinner', 'character', 'yard']]


In [ ]:
w2v = Word2Vec(
    sentences = docs,
    vector_size = 100,
    window = 5,
    min_count = 2,
    sg = 0,
    seed = 42
)

print("Vocabulary size:", len(w2v.wv))

Vocabulary size: 763


In [ ]:
def doc_vector(words):
  known = [w for w in words if w in w2v.wv]
  if len(known) == 0:
    return np.zeroes(100)
  return np.mean(w2v.wv[known], axis = 0)

tweet_vectors = np.array([doc_vector(words) for words in docs])

print(tweet_vectors.shape)


(10000, 100)


In [ ]:
def recommend(query, top_n = 5):
  query_words = clean(query)
  known = [w for w in query_words if w in w2v.wv]

  if len(known) == 0:
    print("No known words in the query:", query)
    return

  query_vector = doc_vector(query_words)
  scores = cosine_similarity([query_vector], tweet_vectors)[0]
  top = np.argsort(scores)[::-1][:top_n]

  print("Query : ", query)
  for i in top:
    print(round(scores[i], 3), df["Text"].iloc[i])
  print()


In [ ]:
recommend(df["Text"].iloc[0])                # tweet 0 should come first
recommend("technology news")                 # may say "No known words"
recommend("zzzz")                            # should say "No known words"

Query :  Party least receive say or single. Prevent prevent husband affect. May himself cup style evening protect. Effect another themselves stage perform.
Possible try tax share style television with. Successful much sell development economy effect.
1.0 Party least receive say or single. Prevent prevent husband affect. May himself cup style evening protect. Effect another themselves stage perform.
Possible try tax share style television with. Successful much sell development economy effect.
1.0 Shake drop research possible agency discuss. Suddenly leg exist. American economy cultural perform.
Every quite situation tax possible. Think watch billion share official pull open.
Music family also ask. Without board main benefit song next.
1.0 Remain that since sister kitchen analysis nor federal. Today free high feel culture over beat day. Page five human food strategy low factor.
Successful once middle. Far information meeting entire. Claim measure opportunity ability sense maybe skin foot